<a href="https://colab.research.google.com/github/venkat123reddy/DeepLearning/blob/main/Assignment3_task_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import tensorflow as tf
import numpy as np

# Load small text chunk (10,000 characters only)
path = tf.keras.utils.get_file("shakespeare.txt",
        "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt")
text = open(path).read()[:10000]

# Character encoding
vocab = sorted(set(text))
char2idx = {u: i for i, u in enumerate(vocab)}
idx2char = np.array(vocab)
text_as_int = np.array([char2idx[c] for c in text])

# Create sequences
seq_length = 50
examples_per_epoch = len(text) // seq_length

char_dataset = tf.data.Dataset.from_tensor_slices(text_as_int)
sequences = char_dataset.batch(seq_length + 1, drop_remainder=True)

def split_input_target(chunk):
    return chunk[:-1], chunk[1:]

dataset = sequences.map(split_input_target)

# Batch and shuffle
BATCH_SIZE = 16
BUFFER_SIZE = 1000
dataset = dataset.shuffle(BUFFER_SIZE).batch(BATCH_SIZE, drop_remainder=True)


1115394/1115394 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [2]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, InputLayer

vocab_size = len(vocab)
embedding_dim = 64
rnn_units = 128

def build_model(vocab_size, embedding_dim, rnn_units, batch_size):
    model = Sequential([
        InputLayer(input_shape=(None,), batch_size=batch_size),
        Embedding(vocab_size, embedding_dim),
        LSTM(rnn_units, return_sequences=True, stateful=True),
        Dense(vocab_size)
    ])
    return model

model = build_model(vocab_size, embedding_dim, rnn_units, BATCH_SIZE)


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


In [3]:
def loss(labels, logits):
    return tf.keras.losses.sparse_categorical_crossentropy(labels, logits, from_logits=True)

model.compile(optimizer='adam', loss=loss)
model.fit(dataset, epochs=5)


Epoch 1/5
12/12 ━━━━━━━━━━━━━━━━━━━━ 3s 37ms/step - loss: 3.9643
Epoch 2/5
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 39ms/step - loss: 3.3665
Epoch 3/5
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - loss: 3.2497
Epoch 4/5
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - loss: 3.2246
Epoch 5/5
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 40ms/step - loss: 3.1715


In [6]:
def generate_text(model, start_string, temperature=1.0, num_generate=300):
    input_eval = [char2idx[c] for c in start_string]
    input_eval = tf.expand_dims(input_eval, 0)
    result = []

    # Reset states of the LSTM layer (assumed to be at index 2 in the model)
    for layer in model.layers:
        if hasattr(layer, 'reset_states'):
            layer.reset_states()

    for _ in range(num_generate):
        predictions = model(input_eval)
        predictions = predictions[:, -1, :] / temperature
        predicted_id = tf.random.categorical(predictions, num_samples=1)[0, 0].numpy()

        result.append(idx2char[predicted_id])
        input_eval = tf.expand_dims([predicted_id], 0)

    return start_string + ''.join(result)
# Build model for generation
gen_model = build_model(vocab_size, embedding_dim, rnn_units, batch_size=1)
gen_model.set_weights(model.get_weights())  # Transfer trained weights

# Generate text
print(generate_text(gen_model, start_string="The king said ", temperature=0.8))



The king said tesoeWslt l nen lireah 
 Poamfeinlst,ns sot:crs tkilEeoy ees   hs nssi r
ss ethsl teshoolpd  ot
:hsucnhdeei  s  Nhntie rW-re s temumge
 hhua
oarimr,essas erecrrl ertesint, ncaTutamfhelo
 hecrn see lrb ukBte teleer hTta
zii rl ue
weetcd p. np ,n oanldew th t eusan me  athmF t lrilis  asetoifiur n  ol
